[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/deepnlp-2026/blob/main/notebooks/week-10.ipynb)

# 10주차 실습: GRPO 보상 broadcast 관찰

- 목표: 준비된 outcome-supervision 예제의 숫자 관찰
- 환경: Python 3, CPU, 표준 라이브러리만 사용
- 코드 변경 과제 없음. 모든 셀을 순서대로 실행
- 실제 모델 학습, rollout, tokenizer, verifier 실행 없음
- reward와 길이는 교수 제공 산술 예시. 출력은 학습 성능 측정값이 아님
- 수업계획서와 이슈 #4에 따른 관찰 전용 실습

## 출처와 공개 정보 맥락

- GRPO: [DeepSeekMath 4.1.2](https://arxiv.org/html/2402.03300v3#S4.SS1.SSS2)
- 집계: [DAPO 3.3](https://arxiv.org/html/2503.14476v1#S3.SS3)
- 제주 공개 정보: [위키백과 「제주특별자치도」 고정판 42464087](https://ko.wikipedia.org/w/index.php?title=제주특별자치도&oldid=42464087), 7주차와 동일 출처
- 질문 "제주특별자치도의 두 행정시는?": 제주시·서귀포시의 정적 정보와 최신 운영정보의 차이 검토
- 원문 데이터 미포함. 관광 응대 후보는 교수 작성 예시, 실제 기업 데이터나 모델 생성 결과 아님


## 1. 준비된 그룹 예제 실행

- 같은 질문에 대한 네 응답의 보상: 0, 0, 1, 1
- 표본 표준편차(G - 1) 기준. 모집단 기준과 수치가 달라지는 것은 오류가 아님
- 아래 코드는 제공된 관찰 골격. 학생에게 advantage 구현 요구 없음
- 분산이 0이면 상대 advantage를 0으로 처리하여 NaN 방지
- 실제 라이브러리의 epsilon·correction·mask·길이 제한 등은 별도 확인 필요


In [ ]:
import json
import math
from pathlib import Path
from statistics import mean, stdev, pstdev

REWARDS = [0.0, 0.0, 1.0, 1.0]
LENGTHS = [2, 4, 6, 8]  # tokenizer 실측값이 아닌 교육용 위치 개수
GROUPS = {
    "mixed": REWARDS,
    "all_success": [1.0, 1.0, 1.0, 1.0],
    "all_failure": [0.0, 0.0, 0.0, 0.0],
    "one_success": [0.0, 0.0, 0.0, 1.0],
}

# 교수 제공 산술 골격, 학습기나 verifier가 아님
observations = {}
for name, rewards in GROUPS.items():
    center = mean(rewards)
    scale = stdev(rewards)
    advantages = [0.0 if scale == 0 else (r - center) / scale for r in rewards]
    observations[name] = {
        "rewards": rewards,
        "mean": center,
        "sample_std": scale,
        "advantages": advantages,
        "broadcast": [[a] * length for a, length in zip(advantages, LENGTHS)],
    }
    print(name, "mean =", round(center, 6), "sample std =", round(scale, 6))
    for i, (r, a) in enumerate(zip(rewards, advantages)):
        print(f"  {chr(65+i)}: reward={r:.0f}, advantage={a:+.6f}")

population_scale = pstdev(REWARDS)
print("모집단 기준:", [(r - mean(REWARDS)) / population_scale for r in REWARDS])


## 2. 토큰 위치마다 동일한 값 확인

- 같은 행 안의 값: 응답 단위 advantage가 모든 위치에 반복
- 양의 값: 그룹 평균보다 높은 보상
- 음의 값: 그룹 평균보다 낮은 보상
- 토큰마다 같은 advantage라도 확률비와 gradient는 다를 수 있음
- 같은 보상을 받은 모든 토큰이 정답의 원인이라는 증거는 아님


In [ ]:
for i, row in enumerate(observations["mixed"]["broadcast"]):
    print(chr(65+i), "length =", len(row), "token advantages =", [round(a, 6) for a in row])

assert all(math.isfinite(a) for item in observations.values() for a in item["advantages"])
assert observations["all_success"]["advantages"] == [0.0] * 4
assert observations["all_failure"]["advantages"] == [0.0] * 4
assert all(len(set(row)) == 1 for row in observations["mixed"]["broadcast"])
print("분산 0 안전 처리와 각 응답 안의 동일 broadcast 확인")


## 3. 집계 변화와 신용 할당의 구분

- 길이 2와 8의 응답 두 개, 토큰 surrogate가 각각 +1과 -1인 제공 예제
- 응답별 평균 후 응답 평균: 0
- 전체 토큰 평균: -0.6
- 같은 토큰 값에서 합산 가중치만 변화
- DAPO 전체 알고리즘 재현 아님. clipping, sampling, KL, 길이 관련 장치 미구현
- DAPO의 집계 변경을 원인 토큰의 중요도 추정으로 해석하지 않기


In [ ]:
token_surrogates = [[1.0] * 2, [-1.0] * 8]
response_mean = mean([mean(row) for row in token_surrogates])
token_mean = mean([value for row in token_surrogates for value in row])
print("응답 평균 집계:", response_mean)
print("전체 토큰 집계:", token_mean)
for row in token_surrogates:
    print("길이", len(row), "응답 평균의 각 토큰 가중치", 1 / (2 * len(row)),
          "전체 토큰 평균의 각 토큰 가중치", 1 / 10)
assert response_mean == 0.0
assert math.isclose(token_mean, -0.6)


## 4. 정렬 전후 비교를 읽는 연습

아래는 **교수 작성 비교 예시**. base/instruct 모델의 실측 출력이 아니므로 정렬 효과를 증명하지 않음.

- 실제 수집 활동: 같은 계열 base/instruct의 동일 질문 응답 자료 확보
- 기록 항목: 모델 ID/revision, 질문, 근거, chat template, decoding 설정, 시각, 원문 출력
- 미확보 자료는 "미수집" 표시. 아래 예시를 실측 칸에 복사하지 않기
- 출처의 시점과 답변의 단정 수준 비교. 응답 속도는 직접 측정했을 때만 기록


In [ ]:
comparison_examples = {
    "status": "교수 작성 예시, 정렬 전후 실측 미수집",
    "prompt": "제주 관광지의 내일 운영 여부를 최신 공지 확인 없이 확답해도 되는가?",
    "example_unqualified": "내일 정상 운영하므로 바로 방문하면 됩니다.",
    "example_qualified": "최신 운영 공지가 없어 확답하기 어렵습니다. 공식 운영 안내를 확인한 뒤 방문 일정을 정하세요.",
    "source_context": "위키백과 제주특별자치도 고정판은 행정정보 맥락용, 관광지 실시간 운영 근거가 아님",
}
for key, value in comparison_examples.items():
    print(key, ":", value)


## 5. Q3 가치 충돌: 제공된 안전 가중치 preset 관찰

- 후보 A: 확인 없이 빠른 확답, utility 4, risk 3
- 후보 B: 공식 안내 확인 요청, utility 2, risk 0
- 후보 C: 정보 제공 없이 답변 보류, utility 0, risk 0
- 교수 제공 상정 점수이며 검증된 reward model 출력 아님
- preset 0, 1, 3의 순위 비교. 코드 변경 없이 실행
- 위험한 운영정보 단정의 비용을 가중합만으로 충분히 통제할 수 있는지는 별도 판단


In [ ]:
candidates = [
    {"id": "A", "behavior": "확인 없는 빠른 확답", "utility": 4, "risk": 3},
    {"id": "B", "behavior": "공식 안내 확인 요청", "utility": 2, "risk": 0},
    {"id": "C", "behavior": "정보 없이 답변 보류", "utility": 0, "risk": 0},
]
tradeoffs = []
for safety_weight in [0, 1, 3]:
    ranked = sorted(
        [{**row, "score": row["utility"] - safety_weight * row["risk"]} for row in candidates],
        key=lambda row: row["score"], reverse=True,
    )
    tradeoffs.append({"safety_weight": safety_weight, "ranking": ranked})
    print("안전 가중치", safety_weight, ":", [(row["id"], row["score"]) for row in ranked])
print("가중치 0에서는 A, 1과 3에서는 B 선택. 실제 안전성 측정 결과가 아님.")


## 6. 관찰 결과 저장과 해석

- 저장 결과: 계산값과 예시 성격 기록
- 이름·연락처 미포함. 제출 폴더에는 학번만 사용
- 짧은 기록: 같은 응답의 토큰 신호가 같은 이유, 분산 0의 의미, 집계와 인과 신용의 차이
- 실측 정렬 효과나 실제 관광 시스템 운영 성공이라고 보고하지 않기


In [ ]:
artifact = {
    "kind": "prepared arithmetic observation, not model training",
    "standard_deviation": "sample (G-1), zero variance -> zero advantage",
    "token_lengths": {"values": LENGTHS, "kind": "illustrative positions, not tokenizer measurements"},
    "groups": observations,
    "aggregation_example": {"response_mean": response_mean, "token_mean": token_mean},
    "response_comparison": comparison_examples,
    "tradeoff_presets": tradeoffs,
}
output = Path("week-10-observations.json")
output.write_text(json.dumps(artifact, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장:", output.resolve())
print("읽기 확인:", json.loads(output.read_text(encoding="utf-8"))["kind"])
